
**Entradas (inputs):** artefactos de `08`, comparación controlada de backbone, barrido/ablación vigente y freeze léxico válido.
# 09b · Cierre formal de modelos en dev

## Objetivo
Cerrar formalmente la selección del mejor modelo en `dev` con una rúbrica multicriterio y generar artefactos de freeze de modelo.

## Rol en la metodología
Este notebook separa la etapa de **decisión/freeze** de la etapa de **comparación de resultados**.

## Entradas
- `data/outputs/barridos_hibridos/<timestamp>/tabla_maestra_comparativa.csv`
- `data/outputs/barridos_hibridos/<timestamp>/ranking_variantes.csv`
- `data/outputs/freeze_lexico_<timestamp>/freeze_lexico_resumen.json`

## Preparación de artefactos
Este notebook puede trabajar de dos formas:
- consumiendo un barrido y un freeze ya existentes;
- o preparando automáticamente un barrido compatible con la corrida base actual y regenerando el freeze léxico antes del cierre.

La preparación automática usa:
- `scripts/ejecutar_barrido_ablacion_hibrido.py`;
- `scripts/audit/generar_freeze_lexico.py`.

## Salidas
- `data/outputs/cierre_modelos_dev_<timestamp>/ranking_modelos_dev.csv`
- `data/outputs/cierre_modelos_dev_<timestamp>/rubrica_seleccion_modelos.csv`
- `data/outputs/cierre_modelos_dev_<timestamp>/decision_modelo_final.md`
- `data/outputs/cierre_modelos_dev_<timestamp>/decision_modelo_final.json`
- `data/outputs/cierre_modelos_dev_<timestamp>/lista_modelos_para_test.json`
- `data/outputs/cierre_modelos_dev_<timestamp>/riesgos_y_limitaciones_dev.md`

## Notebook anterior
- `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb`

## Notebook siguiente
- `notebooks/analysis/09_analisis_errores_hibrido.ipynb`

## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** cierre formal multicriterio del desarrollo en `dev`.
- **Herramientas/librerías:** `pandas`, `json`, artefactos latest/manifests del proyecto y la rúbrica formal de selección implementada también en `scripts/cerrar_modelos_dev.py`.
- **Por qué es adecuada aquí:** evita que la elección final dependa solo de `macro_f1` o de lectura manual de tablas. Obliga a dejar trazabilidad sobre backbone, baseline Transformer, riesgos metodológicos y shortlist para `test`.
- **Limitación:** depende de la calidad y alineación de los artefactos previos; si 08, la comparación de backbone o el error analysis estuvieran desfasados, el cierre sería engañoso.
- **Alternativa peor:** seleccionar el modelo a mano desde tablas sueltas sería menos reproducible y menos defendible.



## Configuración de ejecución
Opcionalmente se puede fijar una corrida base de entrenamiento/features, un barrido o un freeze específicos.
Si se dejan vacíos:
- la corrida base se resuelve automáticamente;
- se busca un barrido compatible con esa corrida;
- y, si no existe, puede prepararse automáticamente antes del cierre.


In [ ]:

BARRIDO_DIR = ''  # Override opcional: data/outputs/barridos_hibridos/<timestamp>
FREEZE_DIR = ''   # Override opcional: data/outputs/freeze_lexico_<timestamp>
TRAIN_RUN_ID_REF = ''      # Override opcional de la corrida base train_<timestamp>
FEATURE_RUN_BASE_REF = ''  # Override opcional de la corrida base fe_<timestamp>
AUTO_PREPARAR_BARRIDO = True
AUTO_REGENERAR_FREEZE = True
BARRIDO_FASES = 'A,B,C'
BARRIDO_TOP_C = 3
TOP_BARRIDO = 30


In [ ]:

from pathlib import Path
import subprocess

from utils_shared import latest_train_run, extract_feature_base_from_train_dir, latest_matching_barrido, latest_freeze_dir

cwd = Path.cwd().resolve()
candidatos = [cwd, *cwd.parents]
repo = next((p for p in candidatos if (p / 'scripts' / 'cerrar_modelos_dev.py').exists()), None)
if repo is None:
    raise RuntimeError('No se pudo resolver la raíz del repositorio desde el notebook actual.')

outputs_dir = repo / 'data' / 'outputs'


def _as_repo_relative(path: Path) -> str:
    try:
        return str(path.relative_to(repo))
    except ValueError:
        return str(path)


train_run_ref = (TRAIN_RUN_ID_REF or '').strip() or latest_train_run(outputs_dir)
if not train_run_ref:
    raise FileNotFoundError('No se encontró ninguna corrida base train_<timestamp> en data/outputs/. Ejecuta 07 primero.')

train_dir_ref = outputs_dir / train_run_ref
if not train_dir_ref.exists():
    raise FileNotFoundError(f'No existe la corrida train seleccionada: {train_dir_ref}')

feature_run_base_ref = (FEATURE_RUN_BASE_REF or '').strip() or extract_feature_base_from_train_dir(train_dir_ref)
if not feature_run_base_ref:
    raise FileNotFoundError(
        'No se pudo inferir la corrida de features desde resumen_entrenamiento.json. '
        'Indica FEATURE_RUN_BASE_REF o vuelve a correr 07 sobre una corrida completa de 06.'
    )

print('TRAIN_RUN_ID_REF  :', train_run_ref)
print('FEATURE_RUN_BASE  :', feature_run_base_ref)

if BARRIDO_DIR:
    barrido_dir = (repo / BARRIDO_DIR).resolve()
    if not barrido_dir.exists():
        raise FileNotFoundError(f'No existe el barrido indicado: {barrido_dir}')
else:
    barrido_dir = latest_matching_barrido(
        outputs_dir,
        ref_train_run=train_run_ref,
        feature_run_base=feature_run_base_ref,
    )
    if barrido_dir is None and AUTO_PREPARAR_BARRIDO:
        cmd_barrido = [
            'python', 'scripts/ejecutar_barrido_ablacion_hibrido.py',
            '--eval-split', 'dev',
            '--feature-run-base', feature_run_base_ref,
            '--ref-train-run', train_run_ref,
            '--fases', BARRIDO_FASES,
            '--top-c', str(BARRIDO_TOP_C),
        ]
        print('No se encontró un barrido compatible. Ejecutando preparación automática:')
        print('$', ' '.join(cmd_barrido))
        proc_barrido = subprocess.run(cmd_barrido, cwd=repo, text=True, capture_output=True)
        print(proc_barrido.stdout)
        if proc_barrido.returncode != 0:
            print(proc_barrido.stderr)
            raise RuntimeError('Falló la preparación automática del barrido híbrido.')
        barrido_dir = latest_matching_barrido(
            outputs_dir,
            ref_train_run=train_run_ref,
            feature_run_base=feature_run_base_ref,
        )

    if barrido_dir is None:
        raise FileNotFoundError(
            'No se encontró un barrido compatible con la corrida base actual. '
            'Indica BARRIDO_DIR manualmente o habilita AUTO_PREPARAR_BARRIDO.'
        )

if FREEZE_DIR:
    freeze_dir = (repo / FREEZE_DIR).resolve()
    if not freeze_dir.exists():
        raise FileNotFoundError(f'No existe el freeze indicado: {freeze_dir}')
else:
    if AUTO_REGENERAR_FREEZE or latest_freeze_dir(outputs_dir) is None:
        cmd_freeze = ['python', 'scripts/audit/generar_freeze_lexico.py']
        print('Generando freeze léxico para el cierre formal:')
        print('$', ' '.join(cmd_freeze))
        proc_freeze = subprocess.run(cmd_freeze, cwd=repo, text=True, capture_output=True)
        print(proc_freeze.stdout)
        if proc_freeze.returncode != 0:
            print(proc_freeze.stderr)
            raise RuntimeError('Falló la generación automática del freeze léxico.')

    freeze_dir = latest_freeze_dir(outputs_dir)
    if freeze_dir is None:
        raise FileNotFoundError('No se encontró ningún freeze léxico en data/outputs/.')

cmd = [
    'python', 'scripts/cerrar_modelos_dev.py',
    '--top-barrido', str(TOP_BARRIDO),
    '--barrido-dir', _as_repo_relative(barrido_dir),
    '--freeze-dir', _as_repo_relative(freeze_dir),
]

print('Barrido usado:', barrido_dir)
print('Freeze usado :', freeze_dir)
print('$', ' '.join(cmd))
proc = subprocess.run(cmd, cwd=repo, text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError('Fallo en el cierre formal de modelos en dev. Revisa el mensaje anterior.')

lineas = [ln.strip() for ln in proc.stdout.splitlines() if ln.strip()]
out_dir = Path(lineas[-1]) if lineas else None
print('Directorio generado:', out_dir)


In [ ]:
if out_dir and out_dir.exists():
    print('Artefactos generados:')
    for p in sorted(out_dir.glob('*')):
        print('-', p.name)
else:
    print('No se pudo resolver el directorio de salida.')
